# 🧩 Mini-Lab: Error Handling & Confidence Scoring

**Module 3: Structured Extraction** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Calculate** confidence scores using log probabilities
2. **Handle** extraction failures gracefully
3. **Implement** retry strategies with fallbacks
4. **Build** robust extraction pipelines

## Target Concepts

| Concept | Description |
|---------|-------------|
| Log probabilities | Token-level confidence scores that indicate model certainty |
| Confidence scoring | Aggregating logprobs to assess overall extraction reliability |
| Error handling | Catching and managing JSON errors, validation errors, and API failures |
| Retry patterns | Strategies for re-attempting failed extractions with exponential backoff |

In [6]:
import json
import math
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel, ValidationError
from typing import Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

from document_schemas import Invoice, Receipt, DOCUMENT_SCHEMAS

def md(text):
    display(Markdown(text))

load_dotenv()
client = OpenAI()

DATA_DIR = Path('../data')

# Load sample documents
sample_invoice = (DATA_DIR / 'invoices' / 'invoice_001_corporate.txt').read_text(encoding='utf-8')

print('✓ Setup complete')

✓ Setup complete


---
## 1. Understanding Log Probabilities

**Log probabilities (logprobs)** tell us how confident the model is about each token it generates.

- Higher logprob (closer to 0) = More confident
- Lower logprob (more negative) = Less confident
- Convert to probability: `prob = exp(logprob)`

In [7]:
def extract_with_logprobs(document: str):
    """
    Extract data and return log probabilities.
    """
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': 'Extract invoice number and total amount as JSON: {"invoice_number": "...", "total": ...}'
            },
            {'role': 'user', 'content': f'Document:\n{document[:1000]}'}
        ],
        response_format={'type': 'json_object'},
        temperature=0,
        logprobs=True,        # Enable log probabilities
        top_logprobs=3        # Get top 3 alternatives per token
    )
    
    return response

# Get response with logprobs
response = extract_with_logprobs(sample_invoice)

print('📤 Extracted content:')
print(response.choices[0].message.content)

print('\n📊 Log probabilities (first 10 tokens):')
logprobs = response.choices[0].logprobs

if logprobs and logprobs.content:
    for i, token_info in enumerate(logprobs.content[:10]):
        prob = math.exp(token_info.logprob)
        print(f'  Token: {repr(token_info.token):15} | logprob: {token_info.logprob:7.3f} | prob: {prob:.1%}')

📤 Extracted content:
{"invoice_number": "INV-2024-0842", "total": 18050.00}

📊 Log probabilities (first 10 tokens):
  Token: '{"'            | logprob:  -0.023 | prob: 97.7%
  Token: 'invoice'       | logprob:   0.000 | prob: 100.0%
  Token: '_number'       | logprob:   0.000 | prob: 100.0%
  Token: '":'            | logprob:  -0.002 | prob: 99.8%
  Token: ' "'            | logprob:   0.000 | prob: 100.0%
  Token: 'INV'           | logprob:  -0.000 | prob: 100.0%
  Token: '-'             | logprob:   0.000 | prob: 100.0%
  Token: '202'           | logprob:   0.000 | prob: 100.0%
  Token: '4'             | logprob:   0.000 | prob: 100.0%
  Token: '-'             | logprob:   0.000 | prob: 100.0%


---
## 2. Calculating Confidence Scores

We can aggregate logprobs to get an overall confidence score.

In [16]:
def calculate_confidence(logprobs) -> float:
    """
    Calculate average confidence from log probabilities.
    
    Returns:
        Confidence score between 0 and 1
    """
    if not logprobs or not logprobs.content:
        return 0.5  # Default if no logprobs
    
    total_prob = 0
    count = 0
    
    for token_info in logprobs.content:
        # Convert log probability to probability
        prob = math.exp(token_info.logprob)
        total_prob += prob
        count += 1
    
    return total_prob / count if count > 0 else 0.5


def calculate_min_confidence(logprobs) -> float:
    """
    Get minimum token confidence (weakest link).
    """
    if not logprobs or not logprobs.content:
        return 0.5
    
    min_prob = 1.0
    for token_info in logprobs.content:
        prob = math.exp(token_info.logprob)
        min_prob = min(min_prob, prob)
    
    return min_prob


# Calculate confidence for our extraction
avg_conf = calculate_confidence(response.choices[0].logprobs)
min_conf = calculate_min_confidence(response.choices[0].logprobs)

print(f'📊 Confidence Scores:')
print(f'  Average confidence: {avg_conf:.1%}')
print(f'  Minimum confidence: {min_conf:.1%}')

📊 Confidence Scores:
  Average confidence: 96.0%
  Minimum confidence: 52.7%


---
## 3. Extraction with Confidence

Combine extraction and confidence scoring.

In [17]:
class ExtractionResult:
    """Result of extraction with metadata."""
    def __init__(self, data, confidence: float, tokens_used: int):
        self.data = data
        self.confidence = confidence
        self.tokens_used = tokens_used
        self.is_reliable = confidence > 0.8  # Threshold


def extract_with_confidence(
    document: str,
    schema: type[BaseModel]
) -> ExtractionResult:
    """
    Extract data with confidence scoring.
    """
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': f'''Extract data according to this schema:
{schema_json}

Return only valid JSON. Use null for uncertain fields.'''
            },
            {'role': 'user', 'content': f'Document:\n{document}'}
        ],
        response_format={'type': 'json_object'},
        temperature=0,
        logprobs=True
    )
    
    # Parse data
    data = json.loads(response.choices[0].message.content)
    validated = schema(**data)
    
    # Calculate confidence
    confidence = calculate_confidence(response.choices[0].logprobs)
    
    # Get token usage
    tokens = response.usage.total_tokens if response.usage else 0
    
    return ExtractionResult(validated, confidence, tokens)


# Test
result = extract_with_confidence(sample_invoice, Invoice)

print('📋 Extraction Result:')
print(f'  Invoice #: {result.data.invoice_number}')
print(f'  Total: ${result.data.total:,.2f}')
print(f'\n📊 Metadata:')
print(f'  Confidence: {result.confidence:.1%}')
print(f'  Reliable: {result.is_reliable}')
print(f'  Tokens used: {result.tokens_used}')

📋 Extraction Result:
  Invoice #: INV-2024-0842
  Total: $24,686.46

📊 Metadata:
  Confidence: 100.0%
  Reliable: True
  Tokens used: 1528


---
## 4. Handling Extraction Failures

Things that can go wrong:
- API errors (rate limits, network issues)
- Invalid JSON (shouldn't happen with JSON mode)
- Validation errors (data doesn't match schema)
- Missing required fields

In [18]:
from dataclasses import dataclass
from enum import Enum

class ExtractionStatus(Enum):
    SUCCESS = 'success'
    VALIDATION_ERROR = 'validation_error'
    JSON_ERROR = 'json_error'
    API_ERROR = 'api_error'
    LOW_CONFIDENCE = 'low_confidence'


@dataclass
class SafeExtractionResult:
    """Extraction result with error handling."""
    status: ExtractionStatus
    data: Optional[BaseModel]
    confidence: float
    error: Optional[str]
    
    @property
    def success(self) -> bool:
        return self.status == ExtractionStatus.SUCCESS


def safe_extract(
    document: str,
    schema: type[BaseModel],
    min_confidence: float = 0.7
) -> SafeExtractionResult:
    """
    Extract with comprehensive error handling.
    """
    try:
        schema_json = json.dumps(schema.model_json_schema(), indent=2)
        
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {
                    'role': 'system',
                    'content': f'Extract data according to schema:\n{schema_json}\nReturn valid JSON only.'
                },
                {'role': 'user', 'content': f'Document:\n{document}'}
            ],
            response_format={'type': 'json_object'},
            temperature=0,
            logprobs=True
        )
        
        # Parse JSON
        try:
            data = json.loads(response.choices[0].message.content)
        except json.JSONDecodeError as e:
            return SafeExtractionResult(
                status=ExtractionStatus.JSON_ERROR,
                data=None,
                confidence=0,
                error=f'JSON parse error: {e}'
            )
        
        # Validate with Pydantic
        try:
            validated = schema(**data)
        except ValidationError as e:
            return SafeExtractionResult(
                status=ExtractionStatus.VALIDATION_ERROR,
                data=None,
                confidence=0,
                error=f'Validation error: {e.error_count()} issues'
            )
        
        # Calculate confidence
        confidence = calculate_confidence(response.choices[0].logprobs)
        
        # Check confidence threshold
        if confidence < min_confidence:
            return SafeExtractionResult(
                status=ExtractionStatus.LOW_CONFIDENCE,
                data=validated,
                confidence=confidence,
                error=f'Confidence {confidence:.1%} below threshold {min_confidence:.1%}'
            )
        
        return SafeExtractionResult(
            status=ExtractionStatus.SUCCESS,
            data=validated,
            confidence=confidence,
            error=None
        )
        
    except Exception as e:
        return SafeExtractionResult(
            status=ExtractionStatus.API_ERROR,
            data=None,
            confidence=0,
            error=f'API error: {e}'
        )


# Test safe extraction
result = safe_extract(sample_invoice, Invoice)

print(f'Status: {result.status.value}')
print(f'Success: {result.success}')
print(f'Confidence: {result.confidence:.1%}')

if result.data:
    print(f'Invoice #: {result.data.invoice_number}')
if result.error:
    print(f'Error: {result.error}')

Status: success
Success: True
Confidence: 100.0%
Invoice #: INV-2024-0842


---
## 5. Retry Strategies

When extraction fails or has low confidence, we can retry with different strategies.

In [19]:
import time

def extract_with_retry(
    document: str,
    schema: type[BaseModel],
    max_retries: int = 3,
    min_confidence: float = 0.7
) -> SafeExtractionResult:
    """
    Extract with retry on failure.
    """
    last_result = None
    
    for attempt in range(max_retries):
        print(f'  Attempt {attempt + 1}/{max_retries}...')
        
        # Adjust temperature for retries (higher = more variety)
        temperature = 0.1 * attempt
        
        result = safe_extract(document, schema, min_confidence)
        last_result = result
        
        if result.success:
            print(f'  ✅ Success on attempt {attempt + 1}')
            return result
        
        # Don't retry on validation errors (won't help)
        if result.status == ExtractionStatus.VALIDATION_ERROR:
            print(f'  ❌ Validation error, not retrying')
            return result
        
        # Wait before retry (exponential backoff)
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f'  ⏳ Waiting {wait_time}s before retry...')
            time.sleep(wait_time)
    
    print(f'  ❌ All {max_retries} attempts failed')
    return last_result


# Test retry (should succeed on first try)
print('🔄 Testing extraction with retry:')
result = extract_with_retry(sample_invoice, Invoice)
print(f'\nFinal status: {result.status.value}')

🔄 Testing extraction with retry:
  Attempt 1/3...
  ✅ Success on attempt 1

Final status: success


---
## 6. Fallback Models

If extraction fails with one model, try a different (often larger) model.

In [20]:
def extract_with_fallback(
    document: str,
    schema: type[BaseModel],
    primary_model: str = 'gpt-4o-mini',
    fallback_model: str = 'gpt-4o',
    min_confidence: float = 0.7
) -> SafeExtractionResult:
    """
    Try primary model first, fallback to larger model if needed.
    """
    print(f'  Trying primary model: {primary_model}')
    
    # Try primary model
    result = safe_extract(document, schema, min_confidence)
    
    if result.success:
        print(f'  ✅ Primary model succeeded')
        return result
    
    # Fallback to larger model for complex cases
    if result.status in [ExtractionStatus.LOW_CONFIDENCE, ExtractionStatus.VALIDATION_ERROR]:
        print(f'  ⚠️ Primary failed ({result.status.value}), trying fallback: {fallback_model}')
        
        # Note: In real code, you'd pass the model to safe_extract
        # For this demo, we'll just show the pattern
        # result = safe_extract(document, schema, min_confidence, model=fallback_model)
        
        print(f'  (Fallback model would be used here)')
    
    return result


# Test fallback pattern
print('🔄 Testing extraction with fallback:')
result = extract_with_fallback(sample_invoice, Invoice)
print(f'\nFinal status: {result.status.value}')

🔄 Testing extraction with fallback:
  Trying primary model: gpt-4o-mini
  ✅ Primary model succeeded

Final status: success


---
## 7. Robust Extractor Class

Let's combine everything into a production-ready extractor.

In [21]:
class RobustExtractor:
    """
    Production-ready document extractor with:
    - Confidence scoring
    - Error handling
    - Retry logic
    - Fallback models
    """
    
    def __init__(
        self,
        client: OpenAI,
        model: str = 'gpt-4o-mini',
        min_confidence: float = 0.7,
        max_retries: int = 2
    ):
        self.client = client
        self.model = model
        self.min_confidence = min_confidence
        self.max_retries = max_retries
    
    def extract(
        self,
        document: str,
        schema: type[BaseModel]
    ) -> SafeExtractionResult:
        """
        Extract with all safety measures.
        """
        for attempt in range(self.max_retries):
            result = self._single_extract(document, schema)
            
            if result.success:
                return result
            
            # Don't retry on validation errors
            if result.status == ExtractionStatus.VALIDATION_ERROR:
                return result
        
        return result
    
    def _single_extract(
        self,
        document: str,
        schema: type[BaseModel]
    ) -> SafeExtractionResult:
        """Single extraction attempt."""
        try:
            schema_json = json.dumps(schema.model_json_schema(), indent=2)
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        'role': 'system',
                        'content': f'Extract data according to schema:\n{schema_json}\nReturn valid JSON.'
                    },
                    {'role': 'user', 'content': f'Document:\n{document}'}
                ],
                response_format={'type': 'json_object'},
                temperature=0,
                logprobs=True
            )
            
            data = json.loads(response.choices[0].message.content)
            validated = schema(**data)
            confidence = calculate_confidence(response.choices[0].logprobs)
            
            if confidence < self.min_confidence:
                return SafeExtractionResult(
                    ExtractionStatus.LOW_CONFIDENCE,
                    validated, confidence,
                    f'Confidence {confidence:.1%} < {self.min_confidence:.1%}'
                )
            
            return SafeExtractionResult(
                ExtractionStatus.SUCCESS,
                validated, confidence, None
            )
            
        except ValidationError as e:
            return SafeExtractionResult(
                ExtractionStatus.VALIDATION_ERROR,
                None, 0, str(e)
            )
        except Exception as e:
            return SafeExtractionResult(
                ExtractionStatus.API_ERROR,
                None, 0, str(e)
            )


# Test robust extractor
extractor = RobustExtractor(client, min_confidence=0.7, max_retries=2)

result = extractor.extract(sample_invoice, Invoice)

print('📋 Robust Extraction Result:')
print(f'  Status: {result.status.value}')
print(f'  Confidence: {result.confidence:.1%}')
if result.data:
    print(f'  Invoice #: {result.data.invoice_number}')
    print(f'  Total: ${result.data.total:,.2f}')

📋 Robust Extraction Result:
  Status: success
  Confidence: 99.9%
  Invoice #: INV-2024-0842
  Total: $24,686.46


## 🎯 Summary

### Key Takeaways

1. **Log probabilities** — enable with `logprobs=True` to get token-level confidence scores
2. **Confidence calculation** — convert logprobs to probabilities with `exp(logprob)` and aggregate
3. **Structured error handling** — distinguish between validation errors, JSON errors, and API errors
4. **Retry strategies** — use exponential backoff for transient errors but skip retries for validation issues
5. **Confidence thresholds** — set minimum confidence levels to flag unreliable extractions

---
## Next Steps

In the final notebook, we'll learn how to:
- **Process documents in batch** efficiently
- **Use async for parallelization**
- **Generate reports** from extractions

→ Continue to **mini-doc-extractor-05-batch-processing.ipynb**